# Parte 1 - Baseline RAG funcional

## Nombres: Jessica Ballesteros, Miguel Álvarez, Darlyn Ludeña

**Implementa el pipeline sobre tu corpus, a partir del Lab-02 del día miércoles:**

In [37]:
import socket
import sys
from pathlib import Path
import sys
import os
from rag_pipeline import RagPipeline
from qdrant_client import QdrantClient
import pandas as pd
import json


In [11]:

# Buscar la raíz del proyecto
raiz = Path.cwd()

while raiz != raiz.parent:
    if (
        (raiz / "pyproject.toml").exists()
        and (raiz / "rag_pipeline.py").exists()
    ):
        break
    raiz = raiz.parent

ROOT = raiz

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Raíz del proyecto:", ROOT)
print("rag_pipeline.py:", (ROOT / "rag_pipeline.py").exists())
print("pyproject.toml:", (ROOT / "pyproject.toml").exists())

Raíz del proyecto: /home/dludena/taller-02-rag-alvarez-ballesteros-ludena
rag_pipeline.py: True
pyproject.toml: True


In [ ]:
# Configuración de la Parte 1

os.environ["EMBEDDING_BACKEND"] = "h200"

# Qdrant NO debe ser :memory: en la Parte 1
os.environ["QDRANT_URL"] = "http://localhost:6333"

# Colección específica del baseline
os.environ["QDRANT_COLLECTION"] = "mmia6013_rag_baseline"

# Corpus real
os.environ["CORPUS_DIR"] = str(ROOT / "corpus")

os.environ["OLLAMA_URL"] = "http://localhost:11434"

print("EMBEDDING_BACKEND:", os.environ["EMBEDDING_BACKEND"])
print("QDRANT_URL:", os.environ["QDRANT_URL"])
print("CORPUS_DIR:", os.environ["CORPUS_DIR"])

EMBEDDING_BACKEND: h200
QDRANT_URL: http://localhost:6333
CORPUS_DIR: /home/dludena/taller-02-rag-alvarez-ballesteros-ludena/corpus


In [13]:


cliente_qdrant = QdrantClient(
    url=os.environ["QDRANT_URL"]
)

print("Colecciones existentes:")
print(cliente_qdrant.get_collections())

Colecciones existentes:
collections=[]


In [14]:
print("Host:", socket.gethostname())


Host: DGX-H200-USFQ


In [15]:
pipeline = RagPipeline(
    collection="mmia6013_rag_baseline"
)

print("Backend:", os.environ["EMBEDDING_BACKEND"])
print("Modelo:", pipeline.encoder.modelo if hasattr(pipeline.encoder, "modelo") else "bge-m3")
print("Tope del modelo:", pipeline.max_seq_tokens)
print("Colección:", pipeline.collection)

/opt/datadog-packages/datadog-apm-library-python/4.11.1/ddtrace_pkgs/site-packages-ddtrace-py3.12-manylinux2014-x86_64/ddtrace/internal/module.py:314: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  self.loader.exec_module(module)


Backend: h200
Modelo: bge-m3
Tope del modelo: 8192
Colección: mmia6013_rag_baseline


#### 1. Ingesta. Parsea los documentos (PDF → texto). Documenta qué se pierde o se degrada (tablas, encabezados,pies de página) y qué documentos, si alguno, la ingesta rechazó por no tener texto útil.

In [16]:
corpus_dir = ROOT / "corpus"

print("Corpus:", corpus_dir)
print("¿Existe?:", corpus_dir.exists())
print("¿Es carpeta?:", corpus_dir.is_dir())

documentos = sorted(corpus_dir.glob("*"))

print("\nDocumentos encontrados:")
for doc in documentos:
    print(" -", doc.name)

Corpus: /home/dludena/taller-02-rag-alvarez-ballesteros-ludena/corpus
¿Existe?: True
¿Es carpeta?: True

Documentos encontrados:
 - .gitkeep
 - bai-2022-constitutional-ai.pdf
 - brown-2020-gpt3.pdf
 - ouyang-2022-instructgpt.pdf
 - rafailov-2023-dpo.pdf
 - vaswani-2017-attention-is-all-you-need.pdf
 - wei-2022-chain-of-thought.pdf


In [17]:
chunks = pipeline.ingest(
    corpus_dir,
    chunk_tokens=512,
    overlap_tokens=102
)

print(f"\nFragmentos obtenidos: {len(chunks)}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (30809 > 8192). Running this sequence through the model will result in indexing errors


ingesta: bai-2022-constitutional-ai.pdf: 34 página(s), 94294 caracteres útiles → 75 fragmento(s)
ingesta: brown-2020-gpt3.pdf: 75 página(s), 186342 caracteres útiles → 156 fragmento(s)
ingesta: ouyang-2022-instructgpt.pdf: 68 página(s), 143167 caracteres útiles → 118 fragmento(s)
ingesta: rafailov-2023-dpo.pdf: 27 página(s), 71700 caracteres útiles → 66 fragmento(s)
ingesta: vaswani-2017-attention-is-all-you-need.pdf: 15 página(s), 31566 caracteres útiles → 25 fragmento(s)
ingesta: wei-2022-chain-of-thought.pdf: 43 página(s), 105503 caracteres útiles → 90 fragmento(s)

Fragmentos obtenidos: 530


In [ ]:


datos_chunks = []

for chunk in chunks:
    datos_chunks.append({
        "chunk_id": chunk.id,
        "documento": chunk.document,
        "chunk_index": chunk.chunk_index,
        "tokens": len(
            pipeline.encoder.tokenizer(
                chunk.text,
                add_special_tokens=False,
                truncation=False
            )["input_ids"]
        ),
        "texto": chunk.text
    })

df_chunks = pd.DataFrame(datos_chunks)
print("Total chunks:", len(df_chunks))

,chunk_id,documento,chunk_index,tokens,texto
0,bai-2022-constitutional-ai-0000,bai-2022-constitutional-ai.pdf,0,512,[page=1] Constitutional AI: Harmlessness from ...
1,bai-2022-constitutional-ai-0001,bai-2022-constitutional-ai.pdf,1,512,"from the finetuned model, use a model to evalu..."
2,bai-2022-constitutional-ai-0002,bai-2022-constitutional-ai.pdf,2,510,"(RL) stage, shown as the sequence of steps at ..."
3,bai-2022-constitutional-ai-0003,bai-2022-constitutional-ai.pdf,3,512,"ying a general AI system, we cannot avoid choo..."
4,bai-2022-constitutional-ai-0004,bai-2022-constitutional-ai.pdf,4,512,+ Harmless Figure 2 We show harmlessness versu...


Total chunks: 530


In [20]:
df_chunks.head()

,chunk_id,documento,chunk_index,tokens,texto
0,bai-2022-constitutional-ai-0000,bai-2022-constitutional-ai.pdf,0,512,[page=1] Constitutional AI: Harmlessness from ...
1,bai-2022-constitutional-ai-0001,bai-2022-constitutional-ai.pdf,1,512,"from the finetuned model, use a model to evalu..."
2,bai-2022-constitutional-ai-0002,bai-2022-constitutional-ai.pdf,2,510,"(RL) stage, shown as the sequence of steps at ..."
3,bai-2022-constitutional-ai-0003,bai-2022-constitutional-ai.pdf,3,512,"ying a general AI system, we cannot avoid choo..."
4,bai-2022-constitutional-ai-0004,bai-2022-constitutional-ai.pdf,4,512,+ Harmless Figure 2 We show harmlessness versu...


In [21]:
carpeta_resultados = ROOT / "resultados" / "parte1"
carpeta_resultados.mkdir(parents=True, exist_ok=True)

archivo_chunks = carpeta_resultados / "numeral1.csv"

df_chunks.to_csv(
    archivo_chunks,
    index=False,
    encoding="utf-8-sig"
)

print(f"Guardado en: {archivo_chunks}")

Guardado en: /home/dludena/taller-02-rag-alvarez-ballesteros-ludena/resultados/parte1/numeral1.csv


### 2. Fragmentación. Baseline de tamaño fijo con solapamiento, en tokens del modelo de embeddings, con `chunk_tokens` y  `overlap_tokens` explícitos. El tope de bge-m3 es 8192 y el  Lab-02 del día miércoles corta por defecto en 512: la justificación dice por qué tu tamaño sirve a tus preguntas, no solo que cabe.

In [22]:
CHUNK_TOKENS = 512
OVERLAP_TOKENS = 102

print("Parámetros de fragmentación")
print("chunk_tokens:", CHUNK_TOKENS)
print("overlap_tokens:", OVERLAP_TOKENS)
print("máximo bge-m3:", pipeline.max_seq_tokens)

Parámetros de fragmentación
chunk_tokens: 512
overlap_tokens: 102
máximo bge-m3: 8192


Se utiliza un tamaño de 512 tokens con 102 tokens de solapamiento. Este tamaño sigue el baseline propuesto por el Lab-02 y mantiene los fragmentos suficientemente por debajo del límite de 8192 tokens de bge-m3. El solapamiento permite conservar parte del contexto cuando una idea se encuentra cerca del límite entre dos fragmentos. La elección será evaluada posteriormente mediante el golden set, por lo que no se asume de antemano que 512 sea óptimo.

Se utiliza un tamaño de 512 tokens con 102 tokens de solapamiento,
siguiendo el tamaño baseline establecido en el Lab-02. Para este corpus,
un fragmento de 512 tokens permite recuperar una unidad de contenido
suficientemente amplia para responder preguntas conceptuales sobre los
papers, manteniendo una granularidad adecuada para distinguir diferentes
temas dentro de un mismo documento. El solapamiento de 102 tokens
(aproximadamente 20 %) permite conservar contexto cuando una idea o
explicación queda dividida entre dos fragmentos. El tamaño se mantiene
muy por debajo del límite de 8192 tokens de bge-m3, pero esta condición
solo garantiza que el fragmento pueda ser procesado; la elección final
será evaluada posteriormente con el golden set mediante Hit Rate y MRR.

###  3. Embedding e indexación. Con bge-m3 en la H200 (EMBEDDING_BACKEND=h200, el valor por defecto, con GlobalProtect conectada) o, sin VPN, con OpenAI (EMBEDDING_BACKEND=openai). Declara en el informe la fila por su id y con su verified_at. Indexa en Qdrant (el contenedor, no :memory:, que es solo para depurar).

In [23]:
pipeline.index(chunks)

print("Índice creado correctamente.")
print("Colección:", pipeline.collection)
print("Número de chunks indexados:", len(chunks))

Índice creado correctamente.
Colección: mmia6013_rag_baseline
Número de chunks indexados: 530


In [25]:
info = cliente_qdrant.get_collection(
    collection_name=pipeline.collection
)

print(info)

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=530 segments_count=8 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prev

In [26]:
cantidad = cliente_qdrant.count(
    collection_name=pipeline.collection,
    exact=True
)

print("Vectores en Qdrant:", cantidad.count)

Vectores en Qdrant: 530


In [27]:
df_embedding = pd.DataFrame([{
    "id": "embed_local_multilingue",
    "proveedor": "BAAI",
    "modelo": "BAAI/bge-m3",
    "dimension": 1024,
    "tokens_secuencia": 8192,
    "multilingue": True,
    "costo_usd_1M": 0,
    "verified_at": "2026-08-27"
}])

display(df_embedding)

,id,proveedor,modelo,dimension,tokens_secuencia,multilingue,costo_usd_1M,verified_at
0,embed_local_multilingue,BAAI,BAAI/bge-m3,1024,8192,True,0,2026-08-27


In [28]:
archivo_embedding = carpeta_resultados / "numeral3_embedding_baseline.csv"

df_embedding.to_csv(
    archivo_embedding,
    index=False,
    encoding="utf-8-sig"
)

print(f"Guardado en: {archivo_embedding}")

Guardado en: /home/dludena/taller-02-rag-alvarez-ballesteros-ludena/resultados/parte1/numeral3_embedding_baseline.csv


### 4. Recuperación. Muestra los top-5 fragmentos para al menos 3 preguntas de prueba, con documento y puntaje.

In [39]:
preguntas_prueba = [
    "What are the main advantages of the Transformer architecture compared with recurrent and convolutional neural networks for sequence transduction?",

    "How does GPT-3 demonstrate that increasing model size and using in-context learning can improve few-shot performance?",

    "How does InstructGPT use human feedback to improve the alignment of language models with user intentions?"
]

In [40]:
resultados_retrieval = []

for pregunta in preguntas_prueba:

    resultados = pipeline.retrieve(
        pregunta,
        top_k=5
    )

    print("=" * 100)
    print("PREGUNTA:")
    print(pregunta)
    print("\nTOP 5 VECINOS:\n")

    for rank, resultado in enumerate(resultados, start=1):

        print(f"--- Vecino {rank} ---")
        print(f"Score:     {resultado['score']:.4f}")
        print(f"Documento: {resultado['document']}")
        print(f"Chunk ID:  {resultado['chunk_id']}")
        print(f"Texto:    {resultado['text']}")
        print()

        resultados_retrieval.append({
            "pregunta": pregunta,
            "rank": rank,
            "score": resultado["score"],
            "documento": resultado["document"],
            "chunk_id": resultado["chunk_id"],
            "texto": resultado["text"]
        })

PREGUNTA:
What are the main advantages of the Transformer architecture compared with recurrent and convolutional neural networks for sequence transduction?

TOP 5 VECINOS:

--- Vecino 1 ---
Score:     0.6861
Documento: vaswani-2017-attention-is-all-you-need.pdf
Chunk ID:  vaswani-2017-attention-is-all-you-need-0003
Texto:    it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as described in section 3.2. Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent 

In [41]:
df_retrieval = pd.DataFrame(resultados_retrieval)

display(df_retrieval)

,pregunta,rank,score,documento,chunk_id,texto
0,What are the main advantages of the Transforme...,1,0.686089,vaswani-2017-attention-is-all-you-need.pdf,vaswani-2017-attention-is-all-you-need-0003,it more difficult to learn dependencies betwee...
1,What are the main advantages of the Transforme...,2,0.670047,vaswani-2017-attention-is-all-you-need.pdf,vaswani-2017-attention-is-all-you-need-0002,"they generate a sequence of hidden statesht, a..."
2,What are the main advantages of the Transforme...,3,0.664278,vaswani-2017-attention-is-all-you-need.pdf,vaswani-2017-attention-is-all-you-need-0017,". During inference, we 9 [page=10] Table 4: Th..."
3,What are the main advantages of the Transforme...,4,0.636401,vaswani-2017-attention-is-all-you-need.pdf,vaswani-2017-attention-is-all-you-need-0018,recurrent layers most commonly used in encoder...
4,What are the main advantages of the Transforme...,5,0.635783,vaswani-2017-attention-is-all-you-need.pdf,vaswani-2017-attention-is-all-you-need-0000,[page=1] Provided proper attribution is provid...
5,How does GPT-3 demonstrate that increasing mod...,1,0.761181,brown-2020-gpt3.pdf,brown-2020-gpt3-0010,"in-context learning abilities. Specifically, w..."
6,How does GPT-3 demonstrate that increasing mod...,2,0.731738,brown-2020-gpt3.pdf,brown-2020-gpt3-0029,PP+20]. GPT-3’s few-shot result further improv...
7,How does GPT-3 demonstrate that increasing mod...,3,0.711648,brown-2020-gpt3.pdf,brown-2020-gpt3-0009,H+20]. Since in-context learning involves abso...
8,How does GPT-3 demonstrate that increasing mod...,4,0.700224,brown-2020-gpt3.pdf,brown-2020-gpt3-0016,"French. In this paper we focus on zero-shot, o..."
9,How does GPT-3 demonstrate that increasing mod...,5,0.690186,brown-2020-gpt3.pdf,brown-2020-gpt3-0011,"example, GPT-3 achieves 81.5 F1 on CoQA in the..."


In [42]:
archivo_retrieval = carpeta_resultados / "numeral4_retrieval_top5.csv"

df_retrieval.to_csv(
    archivo_retrieval,
    index=False,
    encoding="utf-8-sig"
)

print(f"Guardado en: {archivo_retrieval}")

Guardado en: /home/dludena/taller-02-rag-alvarez-ballesteros-ludena/resultados/parte1/numeral4_retrieval_top5.csv


### 5. Generación.`rag_pipeline.generate` ya trae la llamada: con la clave del curso usa la fila propietario_economico; sin clave, Ollama con open_weight_pequeno. El prompt del baseline hace tres cosas—delimita el contexto, obliga a usar solo el contexto y autoriza la abstención con una frase fija— y esa frase es la misma en el prompt, en el golden set y en el detector: ABSTENCION en cambiarla en los tres sitios

In [51]:
resultados_generacion = []

for pregunta in preguntas_prueba:

    resultado = pipeline.answer(
        pregunta,
        top_k=5
    )

    print("=" * 100)
    print("PREGUNTA:")
    print(pregunta)

    print("\nGENERADOR:")
    print(resultado["generator"])

    print("\nABSTENCIÓN:")
    print(resultado["abstained"])

    print("\nRESPUESTA:")
    print(resultado["answer"])

    print("\nPROMPT UTILIZADO:")
    print(resultado["prompt"])

    print()

    resultados_generacion.append({
        "pregunta": resultado["question"],
        "respuesta": resultado["answer"],
        "abstencion": resultado["abstained"],
        "generador": resultado["generator"],
        "prompt": resultado["prompt"]
    })

PREGUNTA:
What are the main advantages of the Transformer architecture compared with recurrent and convolutional neural networks for sequence transduction?

GENERADOR:
ollama:qwen3:32b

ABSTENCIÓN:
False

RESPUESTA:
El Transformer ofrece varias ventajas clave sobre modelos basados en redes recurrentes (RNN) o convolucionales (CNN) para transducción de secuencias:  
1. **Mayor paralelización**: Al eliminar la dependencia secuencial de RNNs/CNNs, el Transformer permite un entrenamiento significativamente más rápido (sección 4 y abstracto).  
2. **Dependencias globales sin restricciones de distancia**: A diferencia de CNNs (donde las operaciones crecen linealmente o logarítmicamente con la distancia entre posiciones), el Transformer relaciona posiciones arbitrarias en una secuencia con un número constante de operaciones, facilitando el aprendizaje de dependencias distantes (contexto 1 y 2).  
3. **Superioridad en rendimiento**: Alcanza resultados de vanguardia en tareas como traducción (2

In [44]:
df_generacion = pd.DataFrame(resultados_generacion)

display(df_generacion[
    [
        "pregunta",
        "respuesta",
        "abstencion",
        "generador"
    ]
])

,pregunta,respuesta,abstencion,generador
0,What are the main advantages of the Transforme...,[modo inspección: no hay clave de API ni Ollam...,None,modo inspección (sin clave ni Ollama)
1,How does GPT-3 demonstrate that increasing mod...,[modo inspección: no hay clave de API ni Ollam...,None,modo inspección (sin clave ni Ollama)
2,How does InstructGPT use human feedback to imp...,[modo inspección: no hay clave de API ni Ollam...,None,modo inspección (sin clave ni Ollama)


In [45]:
archivo_generacion = carpeta_resultados / "numeral5_generacion.csv"

df_generacion.to_csv(
    archivo_generacion,
    index=False,
    encoding="utf-8-sig"
)

print(f"Guardado en: {archivo_generacion}")

Guardado en: /home/dludena/taller-02-rag-alvarez-ballesteros-ludena/resultados/parte1/numeral5_generacion.csv


In [46]:
salidas_generacion = []

for pregunta in preguntas_prueba:

    resultado = pipeline.answer(
        pregunta,
        top_k=5
    )

    salidas_generacion.append(resultado)

archivo_generacion_json = carpeta_resultados / "numeral5_generacion.json"

with open(
    archivo_generacion_json,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        salidas_generacion,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Guardado en: {archivo_generacion_json}")

Guardado en: /home/dludena/taller-02-rag-alvarez-ballesteros-ludena/resultados/parte1/numeral5_generacion.json
